In [ ]:
# Install required libraries
!pip install lpips opencv-python numpy

# Create directories for video uploads
import os
os.makedirs('./videos/traditional', exist_ok=True)
os.makedirs('./videos/sadm', exist_ok=True)
print("Environment setup complete. Directories created successfully.")

Environment setup complete. Directories created successfully.


In [ ]:
import os
import cv2
import torch
import lpips
import numpy as np

# Configuration
VIDEO_DIR_TRADITIONAL = "./videos/traditional"
VIDEO_DIR_SADM = "./videos/sadm"

print("Loading LPIPS model (AlexNet)...")
loss_fn = lpips.LPIPS(net='alex')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loss_fn.to(device)
print(f"Model loaded. Using device: {device}")

def preprocess_image(frame):
    img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = torch.from_numpy(img).float() / 255.0
    img = img * 2.0 - 1.0
    img = img.permute(2, 0, 1).unsqueeze(0)
    return img.to(device)

def calculate_video_lpips(video_path):
    cap = cv2.VideoCapture(video_path)
    ret, prev_frame = cap.read()
    if not ret:
        print(f"Error: Cannot read video {video_path}")
        return []

    prev_tensor = preprocess_image(prev_frame)
    frame_distances = []

    while True:
        ret, curr_frame = cap.read()
        if not ret:
            break

        curr_tensor = preprocess_image(curr_frame)
        with torch.no_grad():
            dist = loss_fn(prev_tensor, curr_tensor).item()

        frame_distances.append(dist)
        prev_tensor = curr_tensor

    cap.release()
    return frame_distances

def process_directory(directory_path, group_name):
    print(f"\nProcessing group: {group_name}")
    print(f"Directory: {directory_path}")
    all_distances = []

    if not os.path.exists(directory_path):
        print(f"Error: Directory {directory_path} does not exist.")
        return

    video_files = [f for f in os.listdir(directory_path) if f.endswith(('.mp4', '.avi', '.mov'))]

    if not video_files:
        print(f"Warning: No video files found in {directory_path}.")
        return

    for video_file in video_files:
        video_path = os.path.join(directory_path, video_file)
        print(f" -> Analyzing {video_file}...", end=" ")

        distances = calculate_video_lpips(video_path)
        if distances:
            vid_mean = np.mean(distances)
            vid_sd = np.std(distances)
            print(f"Done. (Mean LPIPS: {vid_mean:.4f} +- {vid_sd:.4f})")
            all_distances.extend(distances)

    if all_distances:
        total_mean = np.mean(all_distances)
        total_sd = np.std(all_distances)
        print("==========================================")
        print(f"Overall Result for {group_name}:")
        print(f"Mean LPIPS = {total_mean:.4f} +- {total_sd:.4f}")
        print("==========================================\n")
    return total_mean, total_sd

if __name__ == '__main__':
    process_directory(VIDEO_DIR_TRADITIONAL, "Traditional (Control)")
    process_directory(VIDEO_DIR_SADM, "SADM (Experimental)")

Loading LPIPS model (AlexNet)...
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
Model loaded. Using device: cuda

Processing group: Traditional (Control)
Directory: ./videos/traditional
 -> Analyzing Designer_3.mp4... Done. (Mean LPIPS: 0.0108 +- 0.0256)
 -> Analyzing Designer_1.mp4... Done. (Mean LPIPS: 0.0027 +- 0.0198)
 -> Analyzing Designer_2.mp4... Done. (Mean LPIPS: 0.0422 +- 0.0319)
 -> Analyzing Designer_5.mp4... Done. (Mean LPIPS: 0.0185 +- 0.0677)
 -> Analyzing Designer_4.mp4... Done. (Mean LPIPS: 0.0123 +- 0.0161)
Overall Result for Traditional (Control):
Mean LPIPS = 0.0174 +- 0.0379


Processing group: SADM (Experimental)
Directory: ./videos/sadm
 -> Analyzing Design_3.mp4... Done. (Mean LPIPS: 0.0216 +- 0.0030)
 -> Analyzing Design_2.mp4... Done. (Mean LPIPS: 0.1283 +- 0.0918)
 -> Analyzing Design_1.mp4... Done. (Mean LPIPS: 0.0347 +- 0.0089)
 -> Analyzing Desi